# Airline Multi-Step RL Pipeline — Full End-to-End (Colab)

Runs all three phases on a **free Colab T4 GPU** and prints a Before / After reward table.

| Phase | What happens | Model used |
|-------|-------------|------------|
| 0 — Baseline | Scores test split with Groq (Before column) | groq/llama-3.3-70b-versatile |
| 1 — Rollouts | Collects train-split trajectories via Groq | groq/llama-3.3-70b-versatile |
| 2 — RL Train | REINFORCE + LoRA fine-tunes the 7B model | Qwen2.5-7B-Instruct (local) |
| 3 — Eval | Scores test split with tuned model (After col) | tuned weights (local) |

> **Runtime**: `Runtime → Change runtime type → T4 GPU` (free tier).  
> The 7B model needs ~14 GB VRAM in float16; a T4 has 16 GB.

## Cell 1 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → T4 GPU.")
print("GPU:", result.stdout.strip())

## Cell 2 — Install dependencies

Takes ~3 minutes on first run.

In [ ]:
!pip install -q git+https://github.com/sierra-research/tau2-bench.git@airline-tasks
!pip install -q transformers peft accelerate bitsandbytes
!tau2 --help | head -3

## Cell 3 — Set API key

Only Groq is required (free tier, used for rollout collection and the user simulator).  
Get a key at https://console.groq.com

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "gsk_..."   # <-- replace with your key
print("Key set.")

## Cell 4 — Configure (optional)

The defaults are sensible for a T4. Change `BASE_MODEL` only if you have more VRAM.

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────────
# Qwen2.5-7B-Instruct fits a T4 (16 GB) in 4-bit quantisation (~5 GB).
# For float16 (no quantisation) use --no-quantize and need ~14 GB free VRAM.
BASE_MODEL  = "Qwen/Qwen2.5-7B-Instruct"      # change to 0.5B for a quick smoke-test
OUTPUT_DIR  = "output/airline_rl_tuned_7b"    # where tuned weights are saved
MAX_STEPS   = 200                              # RL gradient steps (reduce for speed)

# ── Optional: push tuned model to HuggingFace Hub ───────────────────────────
# Set to "<your-hf-username>/airline-rl-tuned" to upload after training,
# or leave as None to keep weights on disk only.
PUSH_TO_HUB = None

print(f"Base model : {BASE_MODEL}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"RL steps   : {MAX_STEPS}")

## Cell 5 — Patch BASE_MODEL in the script

The script has `BASE_MODEL` hardcoded to 0.5B for local testing.  
This cell patches it to use the 7B model chosen above.

In [ ]:
import importlib, pathlib, re, sys

# Find the installed script
import tau2.scripts.rl_airline_experiment as _script_mod
script_path = pathlib.Path(_script_mod.__file__)

src = script_path.read_text()
patched = re.sub(
    r'^BASE_MODEL\s*=.*$',
    f'BASE_MODEL = "{BASE_MODEL}"',
    src,
    flags=re.MULTILINE,
)
script_path.write_text(patched)

# Reload so imports pick up the new constant
importlib.reload(_script_mod)
print("Patched BASE_MODEL →", _script_mod.BASE_MODEL)

## Cell 6 — Run the full pipeline

This single command runs all four phases and prints the comparison table.  
Expected wall-clock time on a T4:

| Phase | Time |
|-------|------|
| Phase 0 — baseline eval | ~5 min |
| Phase 1 — rollout collection | ~10 min |
| Phase 2 — RL training (200 steps) | ~20 min |
| Phase 3 — post-RL eval | ~15 min |
| **Total** | **~50 min** |

In [ ]:
cmd = [
    "python", "-m", "tau2.scripts.rl_airline_experiment",
    "--model-output-dir", OUTPUT_DIR,
    "--max-steps",        str(MAX_STEPS),
]
if PUSH_TO_HUB:
    cmd += ["--push-to-hub", PUSH_TO_HUB]

print("Running:", " ".join(cmd))
import subprocess, sys
result = subprocess.run(cmd, check=False)
print("\nExit code:", result.returncode)

## Cell 7 — Re-print the comparison table (if needed)

The pipeline prints the table automatically at the end of Cell 6.  
Run this cell to re-read from the saved JSON files at any time.

In [ ]:
import json, pathlib

SIM_DIR = pathlib.Path("data/tau2/simulations")

def load_rewards(glob_pattern: str) -> dict:
    """Load the most recently written file matching the glob."""
    files = sorted(SIM_DIR.glob(glob_pattern), key=lambda p: p.stat().st_mtime)
    if not files:
        return {}
    data = json.loads(files[-1].read_text())
    out = {}
    for sim in data.get("simulations", []) or data if isinstance(data, list) else []:
        tid = sim.get("task_id")
        if "reward" in sim:
            out[tid] = float(sim["reward"])
        elif sim.get("reward_info"):
            out[tid] = float(sim["reward_info"].get("reward", float("nan")))
    print(f"  Loaded {len(out)} tasks from {files[-1].name}")
    return out

before = load_rewards("airline_ms_baseline_test_*.json")
after  = load_rewards("airline_ms_post_eval_*.json")

all_ids = sorted(set(before) | set(after))
print()
print("=" * 55)
print(f"{'Task':<14} {'Before':>12}    {'After':>12}")
print("-" * 55)
bvals, avals = [], []
for tid in all_ids:
    b = before.get(tid)
    a = after.get(tid)
    b_s = f"{b:.4f}" if b is not None else "N/A"
    a_s = f"{a:.4f}" if a is not None else "N/A"
    arrow = " "
    if b is not None and a is not None:
        arrow = "↑" if a > b else ("↓" if a < b else "→")
        bvals.append(b); avals.append(a)
    print(f"{tid:<14} {b_s:>12} {arrow:>4} {a_s:>12}")
print("=" * 55)
if bvals:
    ab, aa = sum(bvals)/len(bvals), sum(avals)/len(avals)
    arrow = "↑" if aa > ab else ("↓" if aa < ab else "→")
    print(f"{'AVERAGE':<14} {ab:>12.4f} {arrow:>4} {aa:>12.4f}")

---

## Notes

### Why 7B and not 0.5B?
The 0.5B model is too small to reliably format tool calls — it generates free-form text
instead of `<tool_call>{...}</tool_call>` blocks, so RL has no signal to improve on.
The 7B model follows the Qwen2.5 tool-call format out of the box.

### Why not vllm?
Phase 3 loads the tuned weights directly with `transformers` — no vllm or external
server required. The tuned model is called via a `LocalHFAgent` subclass that bypasses
litellm entirely.

### Speeding things up
- Reduce `MAX_STEPS` to 50 for a quick smoke-test (~10 min total).
- Use `--filter-failed` to train only on successful episodes (cleaner gradient signal).
- Use `--rl-iterations 3` for on-policy improvement (each iteration re-collects rollouts
  with the latest trained model weights).

### Reusing rollouts from a previous run
If you already collected rollouts locally (`tau2 run --save-to ...`), skip Phase 1:
```bash
python -m tau2.scripts.rl_airline_experiment \
    --skip-rollouts \
    --trajectories-file data/tau2/simulations/<your-file>.json
```